<a href="https://colab.research.google.com/github/anirbanghoshsbi/create_knowledge_base/blob/main/Create_canonical_knowledge_using_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =========================================================
# PHASE B
# CANONICALIZE CLUSTERS USING GPT-4O-MINI
#
# INPUT:
# clustered_concepts.json
#
# OUTPUT:
# canonical_concepts.json
#
# =========================================================


# =========================================================
# INSTALL
# =========================================================

#!pip install -q openai tqdm pydantic requests


# =========================================================
# IMPORTS
# =========================================================

import json
import requests
from collections import defaultdict
from tqdm import tqdm

from openai import OpenAI
from pydantic import BaseModel
from typing import List
from getpass import getpass


# =========================================================
# INPUT API KEY
# =========================================================

api_key = getpass("Enter your OpenAI API Key: ")

client = OpenAI(
    api_key=api_key
)



Enter your OpenAI API Key: ··········


In [3]:

# =========================================================
# DOWNLOAD CLUSTERED CONCEPTS
# =========================================================

URL = "https://raw.githubusercontent.com/anirbanghoshsbi/create_knowledge_base/refs/heads/main/same_as_ever/emdeddings/clustered_concepts.json"

response = requests.get(URL)

clustered_concepts = response.json()

print(f"\nLoaded {len(clustered_concepts)} clustered concepts")


# =========================================================
# GROUP CONCEPTS BY CLUSTER
# =========================================================

clusters = defaultdict(list)

for concept in clustered_concepts:

    cluster_id = concept["cluster_id"]

    # Ignore HDBSCAN noise for now
    if cluster_id == -1:
        continue

    clusters[cluster_id].append(concept)

print(f"Found {len(clusters)} valid clusters")


# =========================================================
# PYDANTIC SCHEMA
# =========================================================

class CanonicalConcept(BaseModel):

    canonical_title: str

    meta_framework: str

    core_principle: str

    implications: List[str]

    tensions: List[str]

    child_concepts: List[str]


# =========================================================
# BUILD CLUSTER TEXT
# =========================================================

def build_cluster_text(cluster_id, concepts):

    text = f"CLUSTER {cluster_id}\n\n"

    for i, c in enumerate(concepts, start=1):

        text += f"""
{i}.

Title:
{c['title']}

Type:
{c['type']}

Statement:
{c['statement']}

"""

    return text


# =========================================================
# CANONICALIZATION FUNCTION
# =========================================================

def canonicalize_cluster(cluster_id, concepts):

    cluster_text = build_cluster_text(
        cluster_id,
        concepts
    )

    prompt = f"""
These concepts belong to the same semantic cluster.

Your task:

1. Create ONE canonical concept.
2. Identify the deepest shared principle.
3. Remove redundancy.
4. Preserve important nuance.
5. Assign a meta-framework category.

IMPORTANT:
- Do NOT invent unrelated abstractions.
- Compress meaning without losing insight.
- Focus on reusable cognitive principles.
- Keep concepts precise and high-signal.

Return JSON only.

Required fields:
- canonical_title
- meta_framework
- core_principle
- implications
- tensions
- child_concepts

CLUSTER:

{cluster_text}
"""

    response = client.beta.chat.completions.parse(

        model="gpt-4o-mini",

        messages=[
            {
                "role": "system",
                "content":
                "You are an expert ontology designer and knowledge compression system."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        response_format=CanonicalConcept
    )

    parsed = response.choices[0].message.parsed

    result = {

        "canonical_id":
        f"CANON_{cluster_id:03d}",

        "canonical_title":
        parsed.canonical_title,

        "meta_framework":
        parsed.meta_framework,

        "core_principle":
        parsed.core_principle,

        "implications":
        parsed.implications,

        "tensions":
        parsed.tensions,

        "child_concepts":
        parsed.child_concepts
    }

    return result


# =========================================================
# PROCESS ALL CLUSTERS
# =========================================================

canonical_concepts = []

for cluster_id, concepts in tqdm(clusters.items()):

    try:

        canonical_node = canonicalize_cluster(
            cluster_id,
            concepts
        )

        canonical_concepts.append(
            canonical_node
        )

        print("\n" + "=" * 60)
        print(f"COMPLETED CLUSTER {cluster_id}")
        print("=" * 60)

        print(
            f"\nCanonical Title:\n"
            f"{canonical_node['canonical_title']}"
        )

    except Exception as e:

        print("\n" + "=" * 60)
        print(f"ERROR IN CLUSTER {cluster_id}")
        print("=" * 60)

        print(e)


# =========================================================
# SAVE OUTPUT
# =========================================================

OUTPUT_FILE = "canonical_concepts.json"

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        canonical_concepts,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 60)
print("PHASE B COMPLETE")
print("=" * 60)

print(
    f"\nSaved canonical concepts to:\n"
    f"{OUTPUT_FILE}"
)


# =========================================================
# SAMPLE OUTPUT
# =========================================================

print("\n" + "=" * 60)
print("SAMPLE CANONICAL NODE")
print("=" * 60)

print(
    json.dumps(
        canonical_concepts[0],
        indent=2,
        ensure_ascii=False
    )
)


Loaded 189 clustered concepts
Found 41 valid clusters


  2%|▏         | 1/41 [00:04<02:48,  4.22s/it]


COMPLETED CLUSTER 39

Canonical Title:
Path Dependency in Historical Processes


  5%|▍         | 2/41 [00:06<01:59,  3.07s/it]


COMPLETED CLUSTER 31

Canonical Title:
Hidden Fragility in Stable Systems


...


COMPLETED CLUSTER 4

Canonical Title:
Attention Dynamics Under Stress


 98%|█████████▊| 40/41 [01:35<00:02,  2.04s/it]


COMPLETED CLUSTER 3

Canonical Title:
Incremental Transformative Change


100%|██████████| 41/41 [01:37<00:00,  2.37s/it]


COMPLETED CLUSTER 33

Canonical Title:
Small Risks and Failures Leading to Catastrophes

PHASE B COMPLETE

Saved canonical concepts to:
canonical_concepts.json

SAMPLE CANONICAL NODE
{
  "canonical_id": "CANON_039",
  "canonical_title": "Path Dependency in Historical Processes",
  "meta_framework": "Causal Dynamics",
  "core_principle": "Minor initial events can significantly influence the trajectory of complex systems through cascading causal chains.",
  "implications": [
    "Small changes in initial conditions can lead to vastly different outcomes over time.",
    "Understanding historical events requires analyzing their interconnected causal origins.",
    "Decision-making is complex due to the unpredictable nature of path dependency."
  ],
  "tensions": [
    "Predictability of outcomes versus randomness of initial events.",
    "Linear versus non-linear interpretations of history.",
    "Individual agency versus systemic constraints in shaping historical trajectories."
  ],
  "c